In [2]:
import pandas as pd
import datetime as dt
from src.data import *
from src.labelling_helpers import *


In [6]:
#select appealCode to process
fnla = "labelled_reports_impacts_laura_v240925.csv"
labelled_laura = pd.read_csv(DATA_LABELLED / fnla)
labelled_laura.loc[labelled_laura["impactSubtype"].isnull(),"impactSubtype"] = labelled_laura["impactSubType"]
labelled_laura = labelled_laura.drop(["impactSubType"],axis=1)
labelled_laura.rename({"impactSubType":"impactSubtype"},inplace=True, axis=1)
fnlu = "labelled_reports_impacts_luca_v230925.csv"
labelled_luca = pd.read_csv(DATA_LABELLED / fnlu)#.drop(["Unnamed: 0"],axis=1)
labelled_luca.rename({"impactSubType":"impactSubtype"},inplace=True, axis=1)
fnga = "labelled_reports_impact_Gabriela_08_2025_fixed_appeal.csv"
labelled_gabi = pd.read_csv(DATA_LABELLED / fnga)#.drop(["Unnamed: 0"],axis=1)
fnana = "labelled_reports_Ana.csv"
labelled_ana = pd.read_csv(DATA_LABELLED / fnana)#.drop(["Unnamed: 0"],axis=1)
labelled_ana.rename({"impactSubType":"impactSubtype"},inplace=True, axis=1)



#reformat to be consistent
labelled_laura["reportDate"] = pd.to_datetime(labelled_laura["reportDate"], dayfirst=False) #reformat date to be consistent
labelled_luca["reportDate"] = pd.to_datetime(labelled_luca["reportDate"], dayfirst=False) #reformat date to be consistent
labelled_gabi["reportDate"] = pd.to_datetime(labelled_gabi["reportDate"], dayfirst=False) #reformat date to be consistent
labelled_ana["reportDate"] = pd.to_datetime(labelled_ana["reportDate"], dayfirst=False) #reformat date to be consistent
labelled_reports = pd.concat([labelled_laura, labelled_luca, labelled_gabi, labelled_ana]).reset_index(drop=True)
labelled_reports.to_csv(DATA_LABELLED / f"labelled_reports_impacts_all_v{dt.datetime.now().strftime('%d%m%y')}.csv", index=False)

In [3]:
# gather extracted reports
ext_df1 = pd.read_csv(DATA_OUT_LLMS / "labelled_reports_turnoff_subtype_val_llama-3.3-70b-versatile_v141025.csv")
ext_df1 = ext_df1[ext_df1["appealCode"]!="MDRYE011"]
ext_df2 = pd.read_csv(DATA_OUT_LLMS / "labelled_reports_turnoff_subtype_val2_llama-3.3-70b-versatile_v141025.csv")
ext_df3 = pd.read_csv(DATA_OUT_LLMS / "labelled_reports_turnoff_subtype_val3_llama-3.3-70b-versatile_v141025.csv")

In [5]:
ext_df_cat = pd.concat([ext_df1, ext_df3,ext_df2])
ext_df_cat.appealCode.unique().shape
ext_df_cat.to_csv(DATA_OUT_LLMS / "labelled_reports_turnoff_subtype_val_all_llama-3.3-70b-versatile_v141025.csv", index=False)

## Check labelling

In [6]:
#load processed reports
file_path = DATA_IN_JSONS / "retest2_new_sel_filtered_report_types_nat_hazards_bugfix_v180925.csv"#'all_ifrc_reports_info_processed_extended_format_nb_std_units.json' "nathaz_ifrc_reports_info_processed.json"
ifrc_reports_df = pd.read_csv(file_path)

In [7]:
#select reports for which labelling has been done
from src.post_process_functions import format_output


labelled_reports = pd.read_csv(DATA_LABELLED / "labelled_reports_impacts_all_v160925.csv")

keys = labelled_reports[['appealCode', 'reportDate']].drop_duplicates()
test_reports = ifrc_reports_df.merge(keys, on=['appealCode', 'reportDate'], how='inner')
print(f"Nb. labeled reports: {len(labelled_reports.appealCode.unique())}, nb. found reports: {len(test_reports)}")
test_reports = format_output(test_reports, list_cols=["sentences","nathaz_text"])


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Nb. labeled reports: 25, nb. found reports: 25


In [12]:
appeals_check = labelled_reports.appealCode.unique().tolist()#labelled_ana.appealCode.unique().tolist() + labelled_gabi.appealCode.unique().tolist() #+ labelled_laura.appealCode.unique() + labelled_luca.appealCode.unique()
iappeal = "MDRSV012"#appeals_check[24]
ireport = test_reports.loc[test_reports.appealCode==iappeal].iloc[0]
download_report(ireport, DATA_PDF)
labelled_reports.loc[labelled_reports.appealCode==iappeal]


,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,appealCode,comments
127,2019-06-26,Affected People,NaN,NaN,NaN,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
128,2019-06-26,Road Infrastructure,24.0,highways,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
129,2019-06-26,Road Infrastructure,31.0,roads,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
130,2019-06-26,Injured People,14.0,people,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
131,2019-06-26,Human Deaths,1.0,people,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
132,2019-06-26,Crop Production and Forestry,42.0,trees,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
133,2019-06-26,Residential Buildings,4.0,Affected homes,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
134,2019-06-26,Residential Buildings,1409.0,Flooded homes,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
135,2019-06-26,Residential Buildings,2.0,Destroyed homes,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN
136,2019-06-26,Informal settlements,13.0,shelters,exact,NaN,NaN,['Situation analysis Description of the disast...,['El Salvador'],"['Morazn department', 'La Union department', '...",2019.0,10.0,7.0,NaN,NaN,NaN,"['Tropical storm', 'Flood', 'Mass Movement']",MDRSV012,NaN


In [11]:
print(iappeal)
ireport.nathaz_text

MDRSV012


['DREF Operation n° MDRSV012 GLIDE: n° TC-2018-000167-SLV Date of issue: 25 June 2019 Date of disaster: 15 October 2018 Operation start date: 1 December 2018 Operation end date: 15 February 2019 DREF allocated: 150,671 Swiss francs (CHF) Number of people affected: 7,085 (1,417 families) Number of people assisted: 2,090 (418 families) Host National Society presence (n° of volunteers, staff, branches): The Salvadorean Red Cross Society (SRCS) has one headquarter, 63 branches throughout the country, 2,239 volunteers and 275 staff.',
 '75 volunteers have been trained as National Intervention Teams (NITs) with different specialties (Water, Sanitation and Hygiene Promotion, Logistics, General, ZIKA and Vector Control, Psychosocial Support (PSS)) and 35 active volunteers trained in the Damage Assessment and Needs Analysis (DANA) assessment tool.',
 'Red Cross Red Crescent Movement partners actively involved in the operation: International Federation of Red Cross and Red Crescent Societies (IF

In [20]:
from importlib import reload
from src import text_processing_functions
reload(text_processing_functions)
from src.text_processing_functions import select_impact_description
select_impact_description(ireport.sentences)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['DREF Operation n° MDRSV012 GLIDE: n° TC-2018-000167-SLV Date of issue: 25 June 2019 Date of disaster: 15 October 2018 Operation start date: 1 December 2018 Operation end date: 15 February 2019 DREF allocated: 150,671 Swiss francs (CHF) Number of people affected: 7,085 (1,417 families) Number of people assisted: 2,090 (418 families) Host National Society presence (n° of volunteers, staff, branches): The Salvadorean Red Cross Society (SRCS) has one headquarter, 63 branches throughout the country, 2,239 volunteers and 275 staff.',
 '75 volunteers have been trained as National Intervention Teams (NITs) with different specialties (Water, Sanitation and Hygiene Promotion, Logistics, General, ZIKA and Vector Control, Psychosocial Support (PSS)) and 35 active volunteers trained in the Damage Assessment and Needs Analysis (DANA) assessment tool.',
 'Red Cross Red Crescent Movement partners actively involved in the operation: International Federation of Red Cross and Red Crescent Societies (IF